# Computer Vision: Interview-Ready Reference

This notebook covers Computer Vision from fundamentals to modern approaches.
Each section builds on the previous, ending with 15 interview questions and answers.

## Table of Contents
1. Image Fundamentals
2. Traditional CV Approaches
3. CNN Architecture Deep Dive
4. Object Detection Concepts
5. Practical Implementation
6. Interview Questions & Answers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec
from sklearn.datasets import load_digits
from sklearn.preprocessing import MinMaxScaler
from scipy import ndimage
from scipy.signal import convolve2d
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

print('Core libraries loaded successfully.')
print(f'NumPy: {np.__version__}')

try:
    import cv2
    print(f'OpenCV available: {cv2.__version__}')
    CV2_AVAILABLE = True
except ImportError:
    print('OpenCV not available -- using scipy/numpy equivalents.')
    CV2_AVAILABLE = False

---
## Section 1: Image Fundamentals

### 1.1 Image as a Matrix

A digital image is simply a multidimensional array of numbers:
- **Grayscale**: 2D array of shape (H, W), values 0-255
- **RGB Color**: 3D array of shape (H, W, 3), one channel per color
- **RGBA**: Shape (H, W, 4), includes alpha (transparency)

**Key interview point**: Images are just numbers. Every CV operation is linear algebra.

In [ ]:
# Load the digits dataset (8x8 grayscale images)
digits = load_digits()
X, y = digits.data, digits.target

print('Digits dataset shape:', X.shape)
print('Each image is', digits.images[0].shape, '-- an 8x8 grayscale matrix')
print('Pixel value range:', X.min(), 'to', X.max())
print('Number of classes:', len(np.unique(y)))

# Show raw matrix values for one image
img = digits.images[0]
print('\nRaw pixel matrix for digit', y[0], ':')
print(np.round(img, 1))

In [ ]:
# Visualize several digits
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(f'Label: {y[i]}')
    ax.axis('off')
plt.suptitle('Sample Images from Digits Dataset (8x8 pixels)', fontsize=14)
plt.tight_layout()
plt.show()

print('Each image is only 64 pixels total -- perfect for seeing CV concepts clearly.')

### 1.2 Simulating RGB Channels

The digits dataset is grayscale, but we can simulate RGB to demonstrate channel concepts.
In a real color image, each channel captures a different frequency of light.

In [ ]:
# Simulate an RGB image from grayscale for demonstration
def make_fake_rgb(gray_img):
    """Create a fake RGB image by modifying each channel slightly."""
    H, W = gray_img.shape
    rgb = np.zeros((H, W, 3), dtype=np.float64)
    rgb[:, :, 0] = gray_img * 1.0   # Red channel
    rgb[:, :, 1] = gray_img * 0.8   # Green channel (dimmer)
    rgb[:, :, 2] = gray_img * 0.5   # Blue channel (dimmest)
    return rgb / rgb.max()           # Normalize to [0, 1]

img_gray = digits.images[0]
img_rgb = make_fake_rgb(img_gray)

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
axes[0].imshow(img_rgb)
axes[0].set_title('Composite RGB')
for c, (name, cmap) in enumerate(zip(['Red', 'Green', 'Blue'],
                                      ['Reds', 'Greens', 'Blues'])):
    axes[c+1].imshow(img_rgb[:, :, c], cmap=cmap)
    axes[c+1].set_title(f'{name} Channel')
for ax in axes:
    ax.axis('off')
plt.suptitle('RGB Channel Decomposition', fontsize=13)
plt.tight_layout()
plt.show()

print('Shape of RGB image:', img_rgb.shape)
print('Access pixel at (3,3):', img_rgb[3, 3])  # [R, G, B] values

### 1.3 Basic Operations: Resize, Crop, Normalize

In [ ]:
from scipy.ndimage import zoom

img = digits.images[3].copy()  # 8x8
print('Original shape:', img.shape)

# Resize: scale up by 4x using nearest-neighbor interpolation
img_upscaled = zoom(img, 4, order=0)   # order=0 = nearest neighbor
img_bilinear = zoom(img, 4, order=1)   # order=1 = bilinear
print('Upscaled shape:', img_upscaled.shape)

# Crop: take top-left 4x4 region
img_crop = img[:4, :4]
print('Cropped shape:', img_crop.shape)

# Normalize to [0, 1]
img_norm = (img - img.min()) / (img.max() - img.min() + 1e-8)
print('Normalized range:', img_norm.min().round(3), 'to', img_norm.max().round(3))

# Standardize (zero mean, unit std)
img_std = (img - img.mean()) / (img.std() + 1e-8)
print('Standardized mean:', img_std.mean().round(4), '  std:', img_std.std().round(4))

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, im, title in zip(axes,
                          [img, img_upscaled, img_crop, img_norm],
                          ['Original (8x8)', 'Upscaled 4x (32x32)', 'Cropped (4x4)', 'Normalized']):
    ax.imshow(im, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

### 1.4 Histogram Equalization

Histogram equalization redistributes pixel intensities to improve contrast.
**Interview point**: It spreads out the most frequent intensity values, making dark images brighter and improving local contrast.

In [ ]:
def histogram_equalization(img):
    """
    Manual histogram equalization.
    Steps:
    1. Compute histogram
    2. Compute CDF (cumulative distribution function)
    3. Map old pixel values to new values using CDF
    """
    img_uint8 = (img / img.max() * 255).astype(np.uint8)
    hist, bins = np.histogram(img_uint8.flatten(), bins=256, range=(0, 256))
    # Compute normalized CDF
    cdf = hist.cumsum()
    cdf_min = cdf[cdf > 0].min()
    n_pixels = img_uint8.size
    # Apply equalization mapping
    cdf_normalized = ((cdf - cdf_min) / (n_pixels - cdf_min) * 255).astype(np.uint8)
    img_eq = cdf_normalized[img_uint8]
    return img_eq, hist, cdf_normalized

img_test = digits.images[1]
img_eq, hist_orig, cdf = histogram_equalization(img_test)

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes[0, 0].imshow(img_test, cmap='gray')
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')
axes[0, 1].imshow(img_eq, cmap='gray')
axes[0, 1].set_title('Histogram Equalized')
axes[0, 1].axis('off')
axes[1, 0].bar(range(256), hist_orig, color='steelblue', alpha=0.7)
axes[1, 0].set_title('Original Histogram')
axes[1, 0].set_xlabel('Pixel Value')
axes[1, 0].set_ylabel('Count')
hist_eq, _ = np.histogram(img_eq.flatten(), bins=256, range=(0, 256))
axes[1, 1].bar(range(256), hist_eq, color='tomato', alpha=0.7)
axes[1, 1].set_title('Equalized Histogram (more uniform)')
axes[1, 1].set_xlabel('Pixel Value')
plt.tight_layout()
plt.show()

print('After equalization, the histogram is more evenly spread across the full range.')

### 1.5 Edge Detection: Sobel and Canny

**Edges** are locations where pixel intensity changes rapidly.
- **Sobel**: Computes gradient magnitude using derivative filters
- **Canny**: Multi-step pipeline -- Gaussian smoothing -> Sobel gradient -> Non-max suppression -> Double threshold -> Hysteresis

**Interview point**: Edges capture structural information while ignoring texture.

In [ ]:
from scipy.ndimage import sobel, gaussian_filter

def sobel_edges(img):
    """Compute edge magnitude using Sobel filters."""
    # Sobel in x and y directions
    sx = sobel(img, axis=1)  # horizontal gradient
    sy = sobel(img, axis=0)  # vertical gradient
    magnitude = np.hypot(sx, sy)
    return magnitude, sx, sy

def simple_canny(img, low_threshold=0.1, high_threshold=0.3, sigma=1.0):
    """
    Simplified Canny-like edge detector.
    Full Canny requires hysteresis; this is a readable approximation.
    """
    # Step 1: Smooth to reduce noise
    smooth = gaussian_filter(img.astype(float), sigma=sigma)
    # Step 2: Compute gradients
    mag, gx, gy = sobel_edges(smooth)
    # Step 3: Normalize
    mag_norm = mag / mag.max()
    # Step 4: Double threshold
    edges = np.zeros_like(mag_norm)
    edges[mag_norm > high_threshold] = 1.0
    edges[(mag_norm >= low_threshold) & (mag_norm <= high_threshold)] = 0.5
    return edges, mag_norm

# Use digit '8' which has interesting edge structure
idx = np.where(y == 8)[0][0]
img_e = digits.images[idx].astype(float)

mag, sx, sy = sobel_edges(img_e)
edges_canny, mag_norm = simple_canny(img_e)

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, im, title in zip(axes,
    [img_e, sx, sy, mag, edges_canny],
    ['Original', 'Sobel-X (horizontal)', 'Sobel-Y (vertical)',
     'Gradient Magnitude', 'Canny-like Edges']):
    ax.imshow(im, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.suptitle('Edge Detection Pipeline', fontsize=13)
plt.tight_layout()
plt.show()

print('Sobel X detects vertical edges (intensity changes left-right).')
print('Sobel Y detects horizontal edges (intensity changes top-bottom).')
print('Gradient magnitude combines both for complete edge map.')

---
## Section 2: Traditional CV Approaches

### 2.1 HOG (Histogram of Oriented Gradients)

HOG describes local appearance and shape of objects by:
1. Dividing image into small cells
2. Computing gradient orientation histogram per cell
3. Normalizing across blocks of cells

**Key properties**:
- Invariant to small translations and rotations
- Captures edge and gradient structure
- Used in pedestrian detection (Dalal & Triggs, 2005)

In [ ]:
def compute_hog_manual(img, cell_size=2, n_bins=8):
    """
    Manual HOG implementation for pedagogical purposes.
    For production use: sklearn.feature_extraction.image or skimage.feature.hog
    """
    H, W = img.shape
    img_f = img.astype(float)

    # Step 1: Compute gradients
    gx = np.zeros_like(img_f)
    gy = np.zeros_like(img_f)
    gx[:, 1:-1] = img_f[:, 2:] - img_f[:, :-2]  # central difference
    gy[1:-1, :] = img_f[2:, :] - img_f[:-2, :]  # central difference

    magnitude = np.sqrt(gx**2 + gy**2)
    orientation = np.degrees(np.arctan2(gy, gx)) % 180  # unsigned [0, 180)

    # Step 2: Build histogram per cell
    cells_y = H // cell_size
    cells_x = W // cell_size
    hog_cells = np.zeros((cells_y, cells_x, n_bins))

    bin_width = 180.0 / n_bins
    for cy in range(cells_y):
        for cx in range(cells_x):
            cell_mag = magnitude[cy*cell_size:(cy+1)*cell_size,
                                 cx*cell_size:(cx+1)*cell_size]
            cell_ori = orientation[cy*cell_size:(cy+1)*cell_size,
                                   cx*cell_size:(cx+1)*cell_size]
            for b in range(n_bins):
                low = b * bin_width
                high = (b + 1) * bin_width
                mask = (cell_ori >= low) & (cell_ori < high)
                hog_cells[cy, cx, b] = cell_mag[mask].sum()

    # Step 3: Flatten to feature vector
    hog_features = hog_cells.flatten()
    # L2 normalize
    norm = np.linalg.norm(hog_features)
    if norm > 0:
        hog_features = hog_features / norm
    return hog_features, hog_cells, magnitude, orientation

# Apply to a digit
img_hog = digits.images[5].astype(float)
features, cells, mag, ori = compute_hog_manual(img_hog)

print(f'Image shape: {img_hog.shape}')
print(f'HOG feature vector length: {len(features)}')
print(f'HOG cells shape: {cells.shape}  (cells_y x cells_x x n_bins)')

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
axes[0].imshow(img_hog, cmap='gray')
axes[0].set_title('Original Image')
axes[0].axis('off')
axes[1].imshow(mag, cmap='hot')
axes[1].set_title('Gradient Magnitude')
axes[1].axis('off')
axes[2].imshow(ori, cmap='hsv')
axes[2].set_title('Gradient Orientation')
axes[2].axis('off')
axes[3].bar(range(len(features)), features, color='steelblue', alpha=0.7)
axes[3].set_title(f'HOG Features ({len(features)}D)')
axes[3].set_xlabel('Feature Index')
plt.suptitle('HOG Feature Extraction', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# HOG with sklearn
try:
    from skimage.feature import hog
    from skimage import exposure
    img_sk = digits.images[0]
    fd, hog_image = hog(img_sk, orientations=8, pixels_per_cell=(2, 2),
                        cells_per_block=(1, 1), visualize=True)
    hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))
    fig, axes = plt.subplots(1, 2, figsize=(8, 3))
    axes[0].imshow(img_sk, cmap='gray')
    axes[0].set_title('Original')
    axes[0].axis('off')
    axes[1].imshow(hog_image_rescaled, cmap='gray')
    axes[1].set_title(f'HOG Visualization ({len(fd)}D features)')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
    print('skimage HOG feature vector length:', len(fd))
except ImportError:
    print('skimage not available. Using manual HOG implementation above.')
    print('Feature vector from manual HOG:', len(features), 'dimensions')

### 2.2 SIFT/SURF Concept — Why We Moved to Deep Learning

**SIFT (Scale-Invariant Feature Transform)**:
- Detects keypoints at multiple scales using Difference of Gaussians
- Describes each keypoint with a 128D histogram of gradients
- **Invariant to**: scale, rotation, illumination, affine distortion

**SURF (Speeded-Up Robust Features)**:
- Faster approximation of SIFT using integral images and box filters

**Why deep learning replaced them**:
1. Handcrafted features require domain expertise and tuning
2. Cannot learn dataset-specific representations
3. Poor performance on complex textures, deformations
4. CNNs learn hierarchical features end-to-end from data
5. At ImageNet scale, deep features outperform SIFT by a large margin

In [ ]:
# Demonstrate the scale-space concept behind SIFT using Gaussian pyramids
def gaussian_pyramid(img, levels=4):
    """Build a Gaussian pyramid -- foundation of scale-space analysis."""
    pyramid = [img.astype(float)]
    for _ in range(levels - 1):
        # Smooth then subsample
        smoothed = gaussian_filter(pyramid[-1], sigma=1.0)
        # Downsample by taking every other pixel (if size allows)
        if smoothed.shape[0] > 2 and smoothed.shape[1] > 2:
            downsampled = smoothed[::2, ::2]
        else:
            downsampled = smoothed
        pyramid.append(downsampled)
    return pyramid

def difference_of_gaussians(img, sigma1=1.0, sigma2=2.0):
    """DoG is the core of SIFT keypoint detection."""
    g1 = gaussian_filter(img.astype(float), sigma=sigma1)
    g2 = gaussian_filter(img.astype(float), sigma=sigma2)
    return g2 - g1

img_sift = digits.images[0].astype(float)
pyramid = gaussian_pyramid(img_sift, levels=3)
dogs = [difference_of_gaussians(img_sift, s, s*1.6)
        for s in [0.5, 1.0, 1.5, 2.0]]

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, (ax, pyr) in enumerate(zip(axes[0], pyramid + [pyramid[-1]])):
    ax.imshow(pyr, cmap='gray')
    ax.set_title(f'Scale {i} ({pyr.shape[0]}x{pyr.shape[1]})')
    ax.axis('off')
for i, (ax, dog) in enumerate(zip(axes[1], dogs)):
    ax.imshow(dog, cmap='RdBu_r')
    ax.set_title(f'DoG sigma={0.5+i*0.5:.1f}')
    ax.axis('off')
plt.suptitle('Scale Space Analysis: Foundation of SIFT', fontsize=13)
plt.tight_layout()
plt.show()
print('SIFT finds keypoints at extrema (min/max) across DoG images at multiple scales.')
print('Each keypoint gets a 128D descriptor -- very powerful but fixed by hand design.')

### 2.3 Image Augmentation Techniques

Augmentation artificially expands training data, improving generalization.
**Key point**: Augmentations should preserve label semantics.

In [ ]:
from scipy.ndimage import rotate as ndimage_rotate
from scipy.ndimage import shift as ndimage_shift

def augment_image(img, mode='flip_h'):
    """Apply various augmentations to a 2D image."""
    img = img.astype(float)
    if mode == 'flip_h':
        return np.fliplr(img)
    elif mode == 'flip_v':
        return np.flipud(img)
    elif mode == 'rotate_15':
        return ndimage_rotate(img, 15, reshape=False, cval=0)
    elif mode == 'rotate_neg15':
        return ndimage_rotate(img, -15, reshape=False, cval=0)
    elif mode == 'shift':
        return ndimage_shift(img, shift=[1, 1], cval=0)
    elif mode == 'brightness_up':
        return np.clip(img * 1.4, 0, 16)
    elif mode == 'brightness_down':
        return np.clip(img * 0.6, 0, 16)
    elif mode == 'noise':
        noise = np.random.normal(0, 0.5, img.shape)
        return np.clip(img + noise, 0, 16)
    elif mode == 'zoom_in':
        zoomed = zoom(img, 1.3)
        # Center-crop back to original size
        h, w = img.shape
        zh, zw = zoomed.shape
        start_h = (zh - h) // 2
        start_w = (zw - w) // 2
        return zoomed[start_h:start_h+h, start_w:start_w+w]
    return img

base_img = digits.images[4].astype(float)
modes = ['flip_h', 'rotate_15', 'rotate_neg15', 'brightness_up',
         'brightness_down', 'noise', 'zoom_in', 'shift']
titles = ['Horiz. Flip', 'Rotate +15', 'Rotate -15', 'Brighter',
          'Darker', 'Add Noise', 'Zoom In', 'Shift']

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes[0, 0].imshow(base_img, cmap='gray')
axes[0, 0].set_title('Original', fontsize=10)
axes[0, 0].axis('off')
for i, (mode, title) in enumerate(zip(modes, titles)):
    row = (i + 1) // 5
    col = (i + 1) % 5
    aug = augment_image(base_img, mode)
    axes[row, col].imshow(aug, cmap='gray')
    axes[row, col].set_title(title, fontsize=10)
    axes[row, col].axis('off')
plt.suptitle('Data Augmentation Techniques', fontsize=13)
plt.tight_layout()
plt.show()

print('Rule of thumb: augmentations that make sense for the task.')
print('Horizontal flip is fine for objects, but bad for digit recognition (6 becomes 9).')

### 2.4 Template Matching

Template matching slides a template over the image and measures similarity at each position.
**Use case**: Find a specific pattern in an image (e.g., find a logo in a scene).

In [ ]:
def template_match_ncc(image, template):
    """
    Normalized Cross-Correlation template matching.
    NCC score of 1 = perfect match, -1 = perfect inverse, 0 = no correlation.
    """
    H, W = image.shape
    th, tw = template.shape
    result = np.zeros((H - th + 1, W - tw + 1))
    tmpl_norm = template - template.mean()
    tmpl_std = template.std()
    if tmpl_std == 0:
        return result
    for i in range(result.shape[0]):
        for j in range(result.shape[1]):
            patch = image[i:i+th, j:j+tw]
            patch_norm = patch - patch.mean()
            patch_std = patch.std()
            if patch_std == 0:
                result[i, j] = 0
            else:
                result[i, j] = np.sum(tmpl_norm * patch_norm) / (th * tw * tmpl_std * patch_std)
    return result

# Create a small synthetic scene and template
np.random.seed(42)
scene = np.random.rand(20, 20) * 4  # noise background
# Plant a bright pattern (a '+' shape) in the scene
scene[7:10, 5:8] = 14  # rectangle
template = scene[7:10, 5:8].copy()  # exact patch as template

corr = template_match_ncc(scene, template)
best_y, best_x = np.unravel_index(np.argmax(corr), corr.shape)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(scene, cmap='gray')
axes[0].set_title('Scene (20x20)')
axes[0].axis('off')
axes[1].imshow(template, cmap='gray')
axes[1].set_title('Template (3x3 patch)')
axes[1].axis('off')
im = axes[2].imshow(corr, cmap='hot')
rect = patches.Rectangle((best_x - 0.5, best_y - 0.5), template.shape[1], template.shape[0],
                           linewidth=2, edgecolor='cyan', facecolor='none')
axes[2].add_patch(rect)
axes[2].set_title(f'NCC Map (max at ({best_x},{best_y}))')
plt.colorbar(im, ax=axes[2])
plt.tight_layout()
plt.show()
print(f'Best match found at position ({best_x}, {best_y}) with NCC = {corr[best_y, best_x]:.4f}')
print('NCC = 1.0 confirms exact match at planted location.')

### 2.5 Why Traditional Features Fail at Scale

| Factor | Traditional (HOG/SIFT) | Deep Learning (CNN) |
|--------|------------------------|---------------------|
| Feature design | Handcrafted by experts | Learned from data |
| Invariance | Limited, specified | Learned automatically |
| Scalability | O(n) features, fixed | Scales with depth |
| Context | No global context | Hierarchical receptive fields |
| Transfer | Domain-specific | Strong general transfer |
| ImageNet top-5 error | ~26% (with SVM) | ~3.6% (ResNet-152) |

---
## Section 3: CNN Architecture Deep Dive

### 3.1 Convolution Operation -- Exact Math

**Convolution** slides a small kernel (filter) over the input and computes a dot product at each position.

For a 2D convolution:
```
(I * K)[i,j] = sum_m sum_n I[i+m, j+n] * K[m, n]
```

**Key parameters**:
- **Kernel size**: e.g., 3x3 or 5x5
- **Stride**: Step size when sliding (larger = smaller output)
- **Padding**: Add zeros around input to control output size
- **Output size**: floor((W - K + 2P) / S) + 1

In [ ]:
# Manual convolution with a small example
def conv2d_manual(img, kernel, stride=1, padding=0):
    """Manual 2D convolution (cross-correlation in practice)."""
    if padding > 0:
        img = np.pad(img, padding, mode='constant', constant_values=0)
    H, W = img.shape
    kh, kw = kernel.shape
    out_h = (H - kh) // stride + 1
    out_w = (W - kw) // stride + 1
    output = np.zeros((out_h, out_w))
    for i in range(0, out_h):
        for j in range(0, out_w):
            patch = img[i*stride:i*stride+kh, j*stride:j*stride+kw]
            output[i, j] = np.sum(patch * kernel)
    return output

# 4x4 input image
I = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]
], dtype=float)

# 2x2 kernel
K = np.array([
    [1, 0],
    [0, -1]
], dtype=float)

result = conv2d_manual(I, K, stride=1, padding=0)
print('Input (4x4):')
print(I)
print('\nKernel (2x2):')
print(K)
print('\nOutput (3x3) from stride=1, no padding:')
print(result)
print('\nOutput size formula: floor((4-2)/1)+1 = 3  (correct)')

# Show common kernels and their effects
kernels = {
    'Edge Detect': np.array([[-1,-1,-1],[-1,8,-1],[-1,-1,-1]]),
    'Sharpen': np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]),
    'Blur (Box)': np.ones((3,3)) / 9.0,
    'Sobel X': np.array([[-1,0,1],[-2,0,2],[-1,0,1]]),
}

test_img = digits.images[0].astype(float)
fig, axes = plt.subplots(1, 5, figsize=(16, 3))
axes[0].imshow(test_img, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')
for i, (name, k) in enumerate(kernels.items()):
    out = convolve2d(test_img, k, mode='same')
    axes[i+1].imshow(out, cmap='gray')
    axes[i+1].set_title(name, fontsize=9)
    axes[i+1].axis('off')
plt.suptitle('Effect of Different Convolutional Kernels', fontsize=13)
plt.tight_layout()
plt.show()

### 3.2 Pooling: Max vs Average

**Max Pooling**: Takes the maximum value in each window
- Preserves the strongest feature activation ("was this feature present?")
- Provides translation invariance
- Most commonly used

**Average Pooling**: Takes the mean of each window
- Smooths feature maps
- Better for tasks needing spatial averaging (global average pooling in classification heads)

In [ ]:
def pool2d(feature_map, pool_size=2, stride=2, mode='max'):
    """2D pooling: max or average."""
    H, W = feature_map.shape
    out_h = (H - pool_size) // stride + 1
    out_w = (W - pool_size) // stride + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            patch = feature_map[i*stride:i*stride+pool_size,
                                j*stride:j*stride+pool_size]
            if mode == 'max':
                output[i, j] = patch.max()
            else:
                output[i, j] = patch.mean()
    return output

# Example: 8x8 feature map -> 4x4 after 2x2 pooling
feature_map = np.array([
    [1, 2, 3, 4, 5, 6, 7, 8],
    [9, 10, 11, 12, 13, 14, 15, 16],
    [17, 18, 3, 2, 1, 4, 2, 3],
    [5, 6, 7, 8, 9, 10, 11, 12],
    [1, 0, 1, 0, 1, 0, 1, 0],
    [0, 1, 0, 1, 0, 1, 0, 1],
    [2, 4, 6, 8, 10, 12, 14, 16],
    [1, 3, 5, 7, 9, 11, 13, 15]
], dtype=float)

max_pooled = pool2d(feature_map, pool_size=2, stride=2, mode='max')
avg_pooled = pool2d(feature_map, pool_size=2, stride=2, mode='average')

print('Feature map shape:', feature_map.shape)
print('After 2x2 max pooling (stride=2):', max_pooled.shape)
print()
print('Original feature map:')
print(feature_map)
print('\nMax pooled (4x4):')
print(max_pooled)
print('\nAverage pooled (4x4):')
print(avg_pooled.round(2))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, data, title in zip(axes,
    [feature_map, max_pooled, avg_pooled],
    ['Feature Map (8x8)', 'Max Pooled (4x4)', 'Avg Pooled (4x4)']):
    im = ax.imshow(data, cmap='YlOrRd')
    ax.set_title(title)
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print('\nMax pooling preserves the strongest activation in each region.')
print('Average pooling retains average signal level -- useful in global pooling.')

### 3.3 Receptive Field Calculation

The **receptive field** is the region of the input image that a single neuron in a feature map can "see".

For stacked convolutions (stride 1, no padding):
- After 1 conv with kernel k: RF = k
- After 2 convs each kernel k: RF = 2k - 1
- General: RF_n = RF_{n-1} + (k-1) * stride_product

With pooling (stride 2):
- Each pooling doubles the receptive field of subsequent layers

In [ ]:
def compute_receptive_field(layers):
    """
    Compute receptive field for a stack of layers.
    layers: list of dicts with 'type', 'kernel', 'stride'
    Returns: list of (layer_name, receptive_field_size)
    """
    rf = 1
    total_stride = 1
    history = [('Input', 1)]
    for layer in layers:
        if layer['type'] in ('conv', 'pool'):
            k = layer['kernel']
            s = layer['stride']
            rf = rf + (k - 1) * total_stride
            total_stride *= s
            history.append((layer['name'], rf))
    return history

# Simulate a VGG-like architecture
vgg_layers = [
    {'type': 'conv', 'name': 'Conv1 (3x3)', 'kernel': 3, 'stride': 1},
    {'type': 'conv', 'name': 'Conv2 (3x3)', 'kernel': 3, 'stride': 1},
    {'type': 'pool', 'name': 'MaxPool1 (2x2)', 'kernel': 2, 'stride': 2},
    {'type': 'conv', 'name': 'Conv3 (3x3)', 'kernel': 3, 'stride': 1},
    {'type': 'conv', 'name': 'Conv4 (3x3)', 'kernel': 3, 'stride': 1},
    {'type': 'pool', 'name': 'MaxPool2 (2x2)', 'kernel': 2, 'stride': 2},
    {'type': 'conv', 'name': 'Conv5 (3x3)', 'kernel': 3, 'stride': 1},
    {'type': 'conv', 'name': 'Conv6 (3x3)', 'kernel': 3, 'stride': 1},
    {'type': 'pool', 'name': 'MaxPool3 (2x2)', 'kernel': 2, 'stride': 2},
]

history = compute_receptive_field(vgg_layers)
print('Receptive Field Growth in VGG-like Architecture:')
print('-' * 45)
for name, rf in history:
    bar = '#' * min(rf, 50)
    print(f'{name:<22} RF = {rf:>4} pixels  {bar}')

plt.figure(figsize=(10, 4))
names = [h[0] for h in history]
rfs = [h[1] for h in history]
plt.plot(rfs, marker='o', color='steelblue', linewidth=2)
plt.xticks(range(len(names)), names, rotation=30, ha='right', fontsize=8)
plt.ylabel('Receptive Field (pixels)')
plt.title('Receptive Field Grows with Depth and Pooling')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nKey insight: Two 3x3 convolutions have the same RF as one 5x5,')
print('but with fewer parameters and more non-linearities -- hence VGG design.')

### 3.4 Classic Architecture Progression

| Architecture | Year | Key Innovation | ImageNet Top-5 Error |
|-------------|------|----------------|----------------------|
| LeNet-5 | 1998 | First CNN for digits, tanh activations | N/A (MNIST) |
| AlexNet | 2012 | ReLU, dropout, GPU training | 15.3% |
| VGGNet | 2014 | Only 3x3 convolutions, depth | 7.3% |
| GoogLeNet | 2014 | Inception modules, 1x1 conv | 6.7% |
| ResNet-152 | 2015 | Skip connections, very deep | 3.57% |
| EfficientNet | 2019 | Neural architecture search, compound scaling | ~1.8% |

In [ ]:
# Compare parameter counts across architectures
architectures = {
    'LeNet-5': {'params_M': 0.06, 'depth': 5, 'year': 1998},
    'AlexNet': {'params_M': 60.0, 'depth': 8, 'year': 2012},
    'VGG-16': {'params_M': 138.0, 'depth': 16, 'year': 2014},
    'GoogLeNet': {'params_M': 6.8, 'depth': 22, 'year': 2014},
    'ResNet-50': {'params_M': 25.6, 'depth': 50, 'year': 2015},
    'ResNet-152': {'params_M': 60.2, 'depth': 152, 'year': 2015},
    'EfficientNet-B0': {'params_M': 5.3, 'depth': 82, 'year': 2019},
}

names = list(architectures.keys())
params = [v['params_M'] for v in architectures.values()]
depths = [v['depth'] for v in architectures.values()]
years = [v['year'] for v in architectures.values()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(names)))
bars = ax1.bar(names, params, color=colors)
ax1.set_ylabel('Parameters (Millions)')
ax1.set_title('Parameter Count by Architecture')
ax1.set_xticklabels(names, rotation=35, ha='right', fontsize=9)
for bar, p in zip(bars, params):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{p}M', ha='center', va='bottom', fontsize=8)

scatter = ax2.scatter(years, depths, s=[p*2 for p in params],
                      c=params, cmap='plasma', alpha=0.7)
for name, year, depth in zip(names, years, depths):
    ax2.annotate(name, (year, depth), textcoords='offset points',
                 xytext=(5, 5), fontsize=7)
ax2.set_xlabel('Year')
ax2.set_ylabel('Network Depth (layers)')
ax2.set_title('Depth Growth Over Time\n(bubble size = # parameters)')
plt.colorbar(scatter, ax=ax2, label='Params (M)')
plt.tight_layout()
plt.show()

### 3.5 ResNet Skip Connections -- Solving Vanishing Gradient

In [ ]:
# Demonstrate how vanishing gradient manifests WITHOUT skip connections
np.random.seed(42)

def simulate_gradient_flow(n_layers, has_skip=False, init_scale=0.5):
    """
    Simulate gradient magnitude as it flows backward through layers.
    Without skip: gradient multiplied by weight norm each layer.
    With skip: gradient has an additive identity path.
    """
    gradient = 1.0
    gradient_history = [gradient]
    for i in range(n_layers):
        # Weight matrix spectral norm (simulated)
        weight_norm = np.random.uniform(0.3, 0.7) * init_scale
        # ReLU derivative (50% chance of passing gradient)
        relu_gate = np.random.uniform(0.4, 0.6)
        grad_through_weights = gradient * weight_norm * relu_gate
        if has_skip:
            # Skip connection adds identity gradient (1.0 from shortcut)
            gradient = grad_through_weights + 0.5  # shortcut adds gradient
        else:
            gradient = grad_through_weights
        gradient_history.append(gradient)
    return gradient_history

np.random.seed(42)
N = 50
no_skip = simulate_gradient_flow(N, has_skip=False)
with_skip = simulate_gradient_flow(N, has_skip=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].semilogy(no_skip, color='tomato', linewidth=2, label='No skip connections')
axes[0].set_xlabel('Layer (from output)')
axes[0].set_ylabel('Gradient Magnitude (log scale)')
axes[0].set_title('Vanishing Gradient WITHOUT Skip Connections')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].plot(with_skip, color='steelblue', linewidth=2, label='With skip connections')
axes[1].set_xlabel('Layer (from output)')
axes[1].set_ylabel('Gradient Magnitude')
axes[1].set_title('Stable Gradient WITH Skip Connections (ResNet)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.suptitle('How ResNet Solves Vanishing Gradient', fontsize=13)
plt.tight_layout()
plt.show()

print('ResNet residual block formula:')
print('  Output = F(x, W) + x')
print('  Gradient: dL/dx = dL/dOutput * (dF/dx + I)')
print("  The identity 'I' term ensures gradient always flows, even if F(x,W) saturates.")
print()
print(f'Without skip: gradient at layer 0 = {no_skip[-1]:.2e}')
print(f'With skip:    gradient at layer 0 = {with_skip[-1]:.2f}')

---
## Section 4: Object Detection Concepts

### 4.1 Sliding Window -- Why It's Slow

In [ ]:
def count_sliding_windows(img_h, img_w, window_sizes, strides):
    """Count how many classifier calls sliding window requires."""
    total = 0
    breakdown = []
    for (wh, ww), stride in zip(window_sizes, strides):
        ny = (img_h - wh) // stride + 1
        nx = (img_w - ww) // stride + 1
        n = ny * nx
        total += n
        breakdown.append((f'{wh}x{ww}', n))
    return total, breakdown

# Typical detection setup: 640x640 image, 3 scales, stride 8
window_sizes = [(32,32),(64,64),(128,128),(256,256),(512,512)]
strides = [8, 16, 32, 64, 128]
total, breakdown = count_sliding_windows(640, 640, window_sizes, strides)

print('Sliding Window Complexity on 640x640 image:')
print(f'{"Window Size":<15} {"# Windows":>12}')
print('-' * 30)
for name, n in breakdown:
    print(f'{name:<15} {n:>12,}')
print('-' * 30)
print(f'Total classifier calls: {total:,}')
print()
print('At 10ms/inference: total time = {:.1f}s per image!'.format(total * 0.01))
print('YOLO processes the whole image ONCE in ~25ms on GPU.')

fig, ax = plt.subplots(figsize=(8, 4))
names = [b[0] for b in breakdown]
counts = [b[1] for b in breakdown]
bars = ax.bar(names, counts, color='steelblue', alpha=0.8)
ax.set_ylabel('Number of Windows')
ax.set_xlabel('Window Size')
ax.set_title('Sliding Window: Classifier Calls per Scale')
for bar, c in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{c:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

### 4.2 Anchor Boxes Concept

Anchor boxes (default boxes) are pre-defined bounding boxes of various aspect ratios and scales.
The model predicts **offsets** from these anchors rather than absolute coordinates.

**Why anchors work**:
- Objects tend to have predictable aspect ratios (cars are wider than tall, people are taller than wide)
- Prediction of small offsets is easier than predicting absolute coordinates
- Multiple anchors per cell allows detecting multiple objects

In [ ]:
def generate_anchors(image_size, scales, aspect_ratios, feature_stride):
    """
    Generate anchor boxes for a given feature map stride.
    Returns list of (x_center, y_center, width, height) in image coordinates.
    """
    anchors = []
    W, H = image_size
    # Grid positions on the feature map
    grid_x = np.arange(feature_stride // 2, W, feature_stride)
    grid_y = np.arange(feature_stride // 2, H, feature_stride)
    for cy in grid_y:
        for cx in grid_x:
            for scale in scales:
                for ratio in aspect_ratios:
                    w = scale * np.sqrt(ratio)
                    h = scale / np.sqrt(ratio)
                    anchors.append((cx, cy, w, h))
    return anchors

anchors = generate_anchors(
    image_size=(320, 320),
    scales=[32, 64, 128],
    aspect_ratios=[0.5, 1.0, 2.0],
    feature_stride=64
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlim(0, 320)
ax.set_ylim(0, 320)
ax.set_aspect('equal')
ax.set_facecolor('#f8f8f8')
cmap = plt.cm.Set1

# Show anchors at center cell only for clarity
center_anchors = [(cx, cy, w, h) for cx, cy, w, h in anchors if cx == 160 and cy == 160]
colors_list = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'pink', 'gray', 'olive']
for i, (cx, cy, w, h) in enumerate(center_anchors[:9]):
    x1, y1 = cx - w/2, cy - h/2
    rect = patches.Rectangle((x1, y1), w, h,
                               linewidth=2, edgecolor=colors_list[i % len(colors_list)],
                               facecolor='none', label=f'Scale {int(w//32)*32}, ratio {h/w:.1f}')
    ax.add_patch(rect)
ax.plot(160, 160, 'k+', markersize=15, markeredgewidth=3)
ax.set_xlabel('Image Width')
ax.set_ylabel('Image Height')
ax.set_title('Anchor Boxes at Center Grid Cell\n(9 anchors: 3 scales x 3 aspect ratios)')
ax.legend(loc='upper right', fontsize=7, ncol=1)
ax.invert_yaxis()
plt.tight_layout()
plt.show()
print(f'Total anchors for 320x320 image with stride=64: {len(anchors)}')
print('For each anchor, model predicts: [dx, dy, dw, dh, objectness, class_scores]')

### 4.3 IoU (Intersection over Union) -- Implement and Visualize

**IoU** measures overlap between predicted and ground-truth bounding boxes.
```
IoU = Area(Intersection) / Area(Union)
```
- IoU = 1.0: Perfect overlap
- IoU = 0.0: No overlap
- Typical threshold: IoU >= 0.5 considered a 'correct' detection

In [ ]:
def compute_iou(box1, box2):
    """
    Compute IoU between two boxes.
    Boxes in format [x1, y1, x2, y2] (top-left, bottom-right).
    """
    # Intersection coordinates
    ix1 = max(box1[0], box2[0])
    iy1 = max(box1[1], box2[1])
    ix2 = min(box1[2], box2[2])
    iy2 = min(box1[3], box2[3])

    inter_w = max(0, ix2 - ix1)
    inter_h = max(0, iy2 - iy1)
    intersection = inter_w * inter_h

    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection

    if union == 0:
        return 0.0
    return intersection / union

def plot_iou(ax, box_gt, box_pred, title):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.set_aspect('equal')
    for box, color, label in [(box_gt, 'green', 'Ground Truth'),
                               (box_pred, 'red', 'Prediction')]:
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                   linewidth=2, edgecolor=color,
                                   facecolor=color, alpha=0.2, label=label)
        ax.add_patch(rect)
    iou = compute_iou(box_gt, box_pred)
    ax.set_title(f'{title}\nIoU = {iou:.3f}', fontsize=10)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.invert_yaxis()

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
scenarios = [
    ([1,1,5,5], [1,1,5,5], 'Perfect Match'),
    ([1,1,5,5], [3,3,7,7], 'Partial Overlap'),
    ([1,1,3,3], [6,6,9,9], 'No Overlap'),
]
for ax, (gt, pred, title) in zip(axes, scenarios):
    plot_iou(ax, gt, pred, title)
plt.suptitle('IoU Visualization', fontsize=13)
plt.tight_layout()
plt.show()

print('IoU examples:')
for gt, pred, title in scenarios:
    iou = compute_iou(gt, pred)
    print(f'  {title}: IoU = {iou:.4f}')

### 4.4 Non-Maximum Suppression (NMS)

Object detectors often produce multiple overlapping boxes for the same object.
NMS removes redundant detections by keeping only the highest-confidence box
among overlapping ones.

In [ ]:
def non_maximum_suppression(boxes, scores, iou_threshold=0.5):
    """
    NMS algorithm:
    1. Sort detections by confidence score (highest first)
    2. Take the highest-scoring box, add to kept list
    3. Remove all boxes that overlap with it (IoU > threshold)
    4. Repeat until no boxes remain
    Returns indices of kept boxes.
    """
    if len(boxes) == 0:
        return []

    # Sort by score descending
    order = np.argsort(scores)[::-1]
    kept = []

    while len(order) > 0:
        # Pick best remaining detection
        i = order[0]
        kept.append(i)
        # Compute IoU with all remaining boxes
        remaining_ious = [compute_iou(boxes[i], boxes[j]) for j in order[1:]]
        # Keep only boxes with IoU below threshold
        order = np.array([order[k+1] for k, iou in enumerate(remaining_ious)
                         if iou < iou_threshold])
    return kept

# Simulate 8 detections for a single object
np.random.seed(7)
gt_box = [2, 2, 8, 7]
# Generate boxes near the ground truth with varying scores
def random_box_near(gt, noise=1.5):
    x1 = gt[0] + np.random.uniform(-noise, noise)
    y1 = gt[1] + np.random.uniform(-noise, noise)
    x2 = gt[2] + np.random.uniform(-noise, noise)
    y2 = gt[3] + np.random.uniform(-noise, noise)
    return [min(x1,x2), min(y1,y2), max(x1,x2), max(y1,y2)]

boxes = [random_box_near(gt_box) for _ in range(8)]
scores = np.random.uniform(0.3, 0.99, 8)
# Add one far box (false positive)
boxes.append([1, 6, 4, 10])
scores = np.append(scores, 0.88)

kept_idx = non_maximum_suppression(boxes, scores, iou_threshold=0.4)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, title, show_all in [(axes[0], 'All Detections (before NMS)', True),
                             (axes[1], f'After NMS ({len(kept_idx)} kept)', False)]:
    ax.set_xlim(-1, 12)
    ax.set_ylim(-1, 12)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    # Ground truth
    r = patches.Rectangle((gt_box[0], gt_box[1]),
                            gt_box[2]-gt_box[0], gt_box[3]-gt_box[1],
                            linewidth=3, edgecolor='lime', facecolor='none',
                            label='Ground Truth', linestyle='--')
    ax.add_patch(r)
    indices = range(len(boxes)) if show_all else kept_idx
    for i in indices:
        b = boxes[i]
        color = 'red' if i not in kept_idx else 'steelblue'
        r = patches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                               linewidth=1.5, edgecolor=color, facecolor=color, alpha=0.15)
        ax.add_patch(r)
        ax.text(b[0], b[1], f'{scores[i]:.2f}', fontsize=7, color=color)
    ax.set_title(title)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
plt.suptitle('Non-Maximum Suppression', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Started with {len(boxes)} detections.')
print(f'Kept {len(kept_idx)} after NMS (IoU threshold=0.4): indices {kept_idx}')
print(f'Scores of kept boxes: {[round(scores[i], 3) for i in kept_idx]}')

### 4.5 mAP (Mean Average Precision)

**mAP** is the standard detection metric:
1. For each class, compute Average Precision (AP) = area under Precision-Recall curve
2. Average AP across all classes

**Key thresholds**:
- **PASCAL VOC**: mAP@0.50 (IoU threshold 0.5)
- **COCO**: mAP@[0.50:0.95] (average over 10 IoU thresholds)

In [ ]:
def compute_ap(recall, precision):
    """
    Compute Average Precision using the 11-point interpolation method.
    (PASCAL VOC style)
    """
    ap = 0.0
    for t in np.linspace(0, 1, 11):
        mask = recall >= t
        if mask.any():
            ap += precision[mask].max()
    return ap / 11.0

def precision_recall_curve(y_true, y_scores):
    """
    Compute precision-recall curve sorted by score.
    y_true: binary labels (1=TP, 0=FP)
    y_scores: confidence scores
    """
    order = np.argsort(y_scores)[::-1]
    y_true_sorted = y_true[order]
    tp_cumsum = np.cumsum(y_true_sorted)
    total_positives = y_true.sum()
    precisions = tp_cumsum / (np.arange(len(y_true_sorted)) + 1)
    recalls = tp_cumsum / total_positives
    return recalls, precisions

# Simulate detection results for 2 classes
np.random.seed(42)
n_detections = 50

# Class 1: good detector
scores_c1 = np.random.beta(4, 2, n_detections)  # higher scores for TPs
labels_c1 = (np.random.uniform(0,1,n_detections) < scores_c1 * 0.9).astype(int)

# Class 2: poor detector
scores_c2 = np.random.uniform(0, 1, n_detections)
labels_c2 = (np.random.uniform(0,1,n_detections) < 0.5).astype(int)

rec1, prec1 = precision_recall_curve(labels_c1, scores_c1)
rec2, prec2 = precision_recall_curve(labels_c2, scores_c2)
ap1 = compute_ap(rec1, prec1)
ap2 = compute_ap(rec2, prec2)
mAP = (ap1 + ap2) / 2

fig, ax = plt.subplots(figsize=(8, 6))
ax.step(rec1, prec1, where='post', color='steelblue', linewidth=2,
        label=f'Class 1 (AP={ap1:.3f})')
ax.step(rec2, prec2, where='post', color='tomato', linewidth=2,
        label=f'Class 2 (AP={ap2:.3f})')
ax.fill_between(rec1, prec1, alpha=0.15, color='steelblue', step='post')
ax.fill_between(rec2, prec2, alpha=0.15, color='tomato', step='post')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
ax.set_title(f'Precision-Recall Curves\nmAP = {mAP:.3f} (average of {ap1:.3f} and {ap2:.3f})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Class 1 AP: {ap1:.4f}  (good detector -- high area under curve)')
print(f'Class 2 AP: {ap2:.4f}  (poor detector -- curve drops quickly)')
print(f'mAP: {mAP:.4f}')

### 4.6 YOLO Architecture Overview

**YOLO (You Only Look Once)** key innovations:
1. Treats detection as a single regression problem
2. Divides image into SxS grid; each cell predicts B boxes + class probabilities
3. Uses anchor boxes at multiple scales (YOLOv3+)
4. Single forward pass -- no region proposals needed

**Output tensor shape**: S x S x (B * 5 + C)
- B = number of anchors per cell
- 5 = [tx, ty, tw, th, objectness]
- C = number of classes

In [ ]:
# Simulate YOLO grid output interpretation
def yolo_decode_boxes(output_grid, anchors, img_size=416):
    """
    Decode YOLO raw output to bounding boxes.
    output_grid: (S, S, B*5) -- simplified, no class dimension
    anchors: list of (w, h) anchor dimensions
    Returns: list of (x1, y1, x2, y2, confidence)
    """
    S = output_grid.shape[0]
    B = len(anchors)
    cell_size = img_size / S
    boxes = []
    for cy in range(S):
        for cx in range(S):
            for b, (aw, ah) in enumerate(anchors):
                tx = output_grid[cy, cx, b*5 + 0]
                ty = output_grid[cy, cx, b*5 + 1]
                tw = output_grid[cy, cx, b*5 + 2]
                th = output_grid[cy, cx, b*5 + 3]
                conf = output_grid[cy, cx, b*5 + 4]
                # Decode
                bx = (cx + 1 / (1 + np.exp(-tx))) * cell_size
                by = (cy + 1 / (1 + np.exp(-ty))) * cell_size
                bw = aw * np.exp(tw)
                bh = ah * np.exp(th)
                x1, y1 = bx - bw/2, by - bh/2
                x2, y2 = bx + bw/2, by + bh/2
                objectness = 1 / (1 + np.exp(-conf))  # sigmoid
                boxes.append((x1, y1, x2, y2, objectness))
    return boxes

# Visualize YOLO grid
S = 7  # 7x7 grid
img_size = 448
cell_size = img_size / S

fig, ax = plt.subplots(figsize=(7, 7))
ax.set_xlim(0, img_size)
ax.set_ylim(0, img_size)
ax.set_facecolor('#eef2f7')
ax.invert_yaxis()

# Draw grid
for i in range(S + 1):
    ax.axhline(i * cell_size, color='gray', linewidth=0.8, alpha=0.6)
    ax.axvline(i * cell_size, color='gray', linewidth=0.8, alpha=0.6)

# Simulate 2 ground truth objects
obj1 = [60, 80, 180, 200]   # person
obj2 = [250, 150, 380, 300]  # car
for box, color, label in [(obj1, 'green', 'Person GT'),
                           (obj2, 'orange', 'Car GT')]:
    r = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                           linewidth=3, edgecolor=color, facecolor='none',
                           label=label)
    ax.add_patch(r)
    # Highlight responsible cell (center of GT box)
    cx = int((box[0]+box[2])/2 / cell_size)
    cy = int((box[1]+box[3])/2 / cell_size)
    cell_rect = patches.Rectangle((cx*cell_size, cy*cell_size), cell_size, cell_size,
                                   linewidth=2, edgecolor=color, facecolor=color, alpha=0.3)
    ax.add_patch(cell_rect)
    ax.text((cx+0.1)*cell_size, (cy+0.5)*cell_size, 'resp.\ncell',
            fontsize=7, color=color, va='center')

ax.set_title(f'YOLO {S}x{S} Grid: Responsible Cells Highlighted', fontsize=12)
ax.legend(loc='upper right')
ax.set_xlabel('Image Width (px)')
ax.set_ylabel('Image Height (px)')
plt.tight_layout()
plt.show()

print(f'YOLO {S}x{S} grid: each cell is {cell_size}x{cell_size} pixels')
print(f'Output tensor: {S}x{S}x(B*5 + C) -- each cell responsible for 1 detection')
print('Cell "responsibility": the cell whose center falls inside the GT box predicts it.')

---
## Section 5: Practical Implementation

### 5.1 Image Classification Pipeline with sklearn

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# Load and split
digits = load_digits()
X, y = digits.data, digits.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train size: {X_train.shape}, Test size: {X_test.shape}')

# Pipeline: StandardScaler -> PCA -> SVM
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=32, whiten=True)),
    ('svm', SVC(kernel='rbf', C=5.0, gamma='scale', probability=True))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print('\nClassification Report:')
print(classification_report(y_test, y_pred))

# Cross-validation
cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
print(f'5-fold CV Accuracy: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

In [ ]:
# Compare multiple classifiers
classifiers = {
    'SVM (RBF)': Pipeline([('sc', StandardScaler()),
                            ('pca', PCA(32, whiten=True)),
                            ('clf', SVC(kernel='rbf', C=5))]),
    'SVM (Linear)': Pipeline([('sc', StandardScaler()),
                               ('pca', PCA(32)),
                               ('clf', SVC(kernel='linear', C=1))]),
    'KNN (k=5)': Pipeline([('sc', StandardScaler()),
                            ('clf', KNeighborsClassifier(5))]),
    'Logistic Reg': Pipeline([('sc', StandardScaler()),
                               ('pca', PCA(32)),
                               ('clf', LogisticRegression(max_iter=1000))]),
    'Random Forest': Pipeline([('sc', StandardScaler()),
                                ('clf', RandomForestClassifier(100, random_state=42))]),
}

results = {}
for name, clf in classifiers.items():
    scores = cross_val_score(clf, X, y, cv=5, scoring='accuracy')
    results[name] = scores
    print(f'{name:<18}: {scores.mean():.4f} +/- {scores.std():.4f}')

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
names_list = list(results.keys())
means = [results[n].mean() for n in names_list]
stds = [results[n].std() for n in names_list]
colors_cls = ['steelblue', 'tomato', 'seagreen', 'darkorange', 'mediumpurple']
bars = ax.bar(names_list, means, yerr=stds, capsize=6, color=colors_cls, alpha=0.8)
ax.set_ylabel('5-fold CV Accuracy')
ax.set_ylim(0.85, 1.02)
ax.set_title('Classifier Comparison on Digits Dataset')
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.002,
            f'{m:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

### 5.2 Feature Extraction and Visualization

PCA on raw pixels gives us interpretable components that look like "eigenndigits".
This is the concept behind feature extraction: learn compact representations.

In [ ]:
from sklearn.decomposition import PCA

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=64)
X_pca = pca.fit_transform(X_scaled)

# Visualize top 16 principal components (eigen-digits)
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    component = pca.components_[i].reshape(8, 8)
    ax.imshow(component, cmap='RdBu_r')
    var_pct = pca.explained_variance_ratio_[i] * 100
    ax.set_title(f'PC {i+1}\n({var_pct:.1f}% var)', fontsize=8)
    ax.axis('off')
plt.suptitle('Top 16 Principal Components (Eigen-Digits)', fontsize=13)
plt.tight_layout()
plt.show()

# Scree plot
cumvar = np.cumsum(pca.explained_variance_ratio_)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(range(1, 33), pca.explained_variance_ratio_[:32] * 100,
        color='steelblue', alpha=0.7)
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained Variance (%)')
ax1.set_title('Scree Plot')
ax2.plot(range(1, 65), cumvar * 100, color='tomato', linewidth=2)
ax2.axhline(95, color='gray', linestyle='--', label='95% threshold')
ax2.set_xlabel('Number of Components')
ax2.set_ylabel('Cumulative Explained Variance (%)')
ax2.set_title('Cumulative Explained Variance')
ax2.legend()
for ax in [ax1, ax2]:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

n_for_95 = (cumvar >= 0.95).argmax() + 1
print(f'Components needed for 95% variance: {n_for_95}')
print(f'Compression ratio: {64}/{n_for_95} = {64/n_for_95:.1f}x')

### 5.3 Data Augmentation Pipeline

In [ ]:
def augmentation_pipeline(img, augmentations):
    """
    Apply a sequence of augmentations with configurable probabilities.
    augmentations: list of (name, probability, params_dict)
    """
    result = img.astype(float)
    applied = []
    for name, prob, params in augmentations:
        if np.random.random() < prob:
            if name == 'rotate':
                angle = np.random.uniform(-params['max_angle'], params['max_angle'])
                result = ndimage_rotate(result, angle, reshape=False, cval=0)
            elif name == 'shift':
                dy = np.random.uniform(-params['max_px'], params['max_px'])
                dx = np.random.uniform(-params['max_px'], params['max_px'])
                result = ndimage_shift(result, [dy, dx], cval=0)
            elif name == 'brightness':
                factor = np.random.uniform(params['low'], params['high'])
                result = np.clip(result * factor, 0, result.max())
            elif name == 'noise':
                noise = np.random.normal(0, params['std'], result.shape)
                result = np.clip(result + noise, 0, result.max())
            elif name == 'flip_h':
                result = np.fliplr(result)
            applied.append(name)
    return result, applied

# Define augmentation pipeline
aug_config = [
    ('rotate',     0.7, {'max_angle': 20}),
    ('shift',      0.6, {'max_px': 1.0}),
    ('brightness', 0.5, {'low': 0.7, 'high': 1.4}),
    ('noise',      0.4, {'std': 0.3}),
]

np.random.seed(100)
base = digits.images[7]
fig, axes = plt.subplots(3, 6, figsize=(15, 7))
for i, ax in enumerate(axes.flat):
    if i == 0:
        ax.imshow(base, cmap='gray')
        ax.set_title('Original', fontsize=8)
    else:
        aug, applied = augmentation_pipeline(base, aug_config)
        ax.imshow(aug, cmap='gray')
        ax.set_title('+'.join(applied) if applied else 'no aug', fontsize=6)
    ax.axis('off')
plt.suptitle('Data Augmentation Pipeline: 17 augmented versions', fontsize=12)
plt.tight_layout()
plt.show()
print('Augmentation expands dataset diversity and prevents overfitting.')

### 5.4 Transfer Learning Workflow

Transfer learning uses features from a model trained on a large dataset (ImageNet)
and adapts them to a new, smaller task.

**Three strategies**:
1. **Feature extraction**: Freeze pretrained layers, train only the head
2. **Fine-tuning**: Unfreeze top layers, train with low learning rate
3. **Full fine-tuning**: Train all layers (needs large dataset)

In [ ]:
# Simulate transfer learning with sklearn -- HOG features as 'pretrained features'
try:
    from skimage.feature import hog
    def extract_hog_features(images):
        features = []
        for img in images:
            fd = hog(img, orientations=8, pixels_per_cell=(2, 2),
                     cells_per_block=(1, 1), visualize=False)
            features.append(fd)
        return np.array(features)
    print('Using skimage HOG as pretrained feature extractor...')
except ImportError:
    def extract_hog_features(images):
        """Fallback: use manual gradient features."""
        features = []
        for img in images:
            gx = np.gradient(img, axis=1)
            gy = np.gradient(img, axis=0)
            mag = np.sqrt(gx**2 + gy**2)
            feat = np.concatenate([mag.flatten(), gx.flatten()[:32]])
            features.append(feat)
        return np.array(features)
    print('Using manual gradient features as proxy for pretrained features...')

# Extract features
X_images = digits.images  # (1797, 8, 8)
X_hog = extract_hog_features(X_images)
print(f'HOG feature shape: {X_hog.shape}')

# Transfer learning: use HOG features + simple linear classifier
X_tr, X_te, y_tr, y_te = train_test_split(X_hog, y, test_size=0.2, random_state=42)
clf = LogisticRegression(max_iter=2000, C=1.0)
clf.fit(X_tr, y_tr)
test_acc = clf.score(X_te, y_te)

print(f'Transfer learning accuracy (HOG + LogReg): {test_acc:.4f}')

# Compare: raw pixels baseline
clf_raw = LogisticRegression(max_iter=500, C=1.0)
clf_raw.fit(X_train, y_train)
raw_acc = clf_raw.score(X_test, y_test)
print(f'Raw pixel baseline (LogReg): {raw_acc:.4f}')
print(f'Improvement from "pretrained" features: +{(test_acc - raw_acc)*100:.1f}%')

In [ ]:
# Visualize transfer learning concept
stages = [
    ('Pretrained\nBackbone\n(ImageNet)', 'Block A\n(Low-level)\nEdges, Colors', '#4C72B0'),
    ('', 'Block B\n(Mid-level)\nTextures, Shapes', '#55A868'),
    ('', 'Block C\n(High-level)\nObject Parts', '#C44E52'),
    ('Task-specific\nHead\n(New task)', 'Classifier\n(Fine-tuned)', '#8172B2'),
]
fig, ax = plt.subplots(figsize=(12, 5))
ax.axis('off')
for i, (section, block, color) in enumerate(stages):
    x = 0.1 + i * 0.22
    rect = patches.FancyBboxPatch((x, 0.2), 0.18, 0.6,
                                   boxstyle='round,pad=0.02',
                                   linewidth=2, edgecolor=color,
                                   facecolor=color, alpha=0.25)
    ax.add_patch(rect)
    ax.text(x + 0.09, 0.5, block, ha='center', va='center',
            fontsize=10, color=color, fontweight='bold', wrap=True)
    if section:
        ax.text(x + 0.09, 0.88, section, ha='center', va='bottom',
                fontsize=9, style='italic', color='gray')
    if i < len(stages) - 1:
        ax.annotate('', xy=(x + 0.22, 0.5), xytext=(x + 0.18, 0.5),
                    arrowprops=dict(arrowstyle='->', lw=2, color='black'))

ax.text(0.1, 0.1, 'FREEZE (weights fixed)', fontsize=9, color='steelblue')
ax.text(0.76, 0.1, 'TRAIN (learn from new data)', fontsize=9, color='#8172B2')
ax.axvline(0.74, ymin=0.05, ymax=0.85, color='black', linestyle='--', linewidth=1.5)
ax.set_title('Transfer Learning: Feature Extraction Mode', fontsize=13)
plt.tight_layout()
plt.show()

---
## Section 6: 15 CV Interview Questions with Answers

### Q1: What is the difference between convolution and cross-correlation?

**Answer**:

- **Mathematical convolution**: Flips the kernel 180 degrees before sliding
  `(I * K)[i,j] = sum K[-m,-n] * I[i+m, j+n]`

- **Cross-correlation**: Does NOT flip the kernel (slides it directly)
  `(I * K)[i,j] = sum K[m,n] * I[i+m, j+n]`

In deep learning frameworks (PyTorch, TensorFlow), `Conv2d` actually performs **cross-correlation**, not true convolution. Since filters are learned, the distinction doesn't matter -- the network just learns flipped filters if it needs to. The terms are used interchangeably in the DL community.

**Key point for interview**: CNNs use cross-correlation. Mathematically they're equivalent when filters are learned.

In [ ]:
# Demonstrate the difference
from scipy.signal import correlate2d

test = np.array([[1,2,3],[4,5,6],[7,8,9]], dtype=float)
kernel = np.array([[1,0],[0,-1]], dtype=float)

# True convolution (flip kernel)
kernel_flipped = np.rot90(kernel, 2)
conv_result = convolve2d(test, kernel_flipped, mode='valid')

# Cross-correlation (no flip)
corr_result = correlate2d(test, kernel, mode='valid')

# Direct cross-correlation
xcorr_manual = convolve2d(test, kernel, mode='valid')

print('Input:')
print(test)
print('\nKernel:')
print(kernel)
print('\nTrue Convolution (kernel flipped):')
print(conv_result)
print('\nCross-Correlation (no flip):')
print(corr_result)
print('\nNote: results differ because kernel is asymmetric.')
print('For symmetric kernels (e.g., Gaussian), conv == cross-corr.')

### Q2: Why do we use max pooling instead of average pooling in most CNNs?

**Answer**:

**Max pooling** is preferred because:
1. **Preserves strongest feature**: Answers "was this feature present?" -- perfect for detection
2. **Translation invariance**: A feature slightly shifted still produces same output if max is preserved
3. **Sparser gradients**: Only the winning unit receives gradient (acts as implicit feature selection)
4. **Works well with sparse activations**: After ReLU, many zeros exist; max pooling finds non-zero peaks

**Average pooling** is better when:
- You need global spatial averaging (Global Average Pooling in classification heads -- e.g., GoogLeNet, ResNet)
- Smooth spatial features matter
- Less prone to throwing away information

**Modern trend**: Many architectures (MobileNet, EfficientNet) use Global Average Pooling before the FC layer to reduce parameters, replacing the large FC blocks of AlexNet/VGG.

### Q3: What problem do skip connections in ResNet solve? Explain technically.

**Answer**:

**Two related problems solved**:

1. **Vanishing gradient**: During backprop, gradients are multiplied by weights at every layer. With many layers, gradient magnitudes approach zero exponentially, making early layers learn very slowly. Skip connections create a shortcut path with gradient = 1 (identity), so gradient flows directly to early layers.

2. **Degradation problem**: Deeper networks had *higher* training error than shallower ones (not just test error -- this is NOT overfitting). This is because optimization became harder, not because of capacity. Skip connections let layers learn *residuals* `F(x) = H(x) - x` instead of the full mapping, which is easier when the target mapping is close to identity.

**Mathematical insight**:
- Without skip: `H(x) = F(x, W)` -- learn from scratch
- With skip: `H(x) = F(x, W) + x` -- learn the residual delta
- If `F = 0`, output is just `x` -- identity mapping is trivially representable
- Gradient: `dL/dx = dL/dH * (dF/dx + I)` -- the `+I` term ensures non-zero gradient

**Result**: Networks with 152+ layers became trainable (from 8 before ResNet).

### Q4: How does YOLO detect multiple objects in a single forward pass?

**Answer**:

1. **Grid division**: Image divided into SxS grid (e.g., 7x7 in YOLOv1, 13x13 in YOLOv3 smallest scale)

2. **Per-cell prediction**: Each cell simultaneously predicts B bounding boxes + class probabilities in one shot -- no sequential region proposals

3. **Output tensor**: Single network produces tensor of shape `S x S x (B*5 + C)`
   - B boxes per cell, each with (x, y, w, h, confidence)
   - C class probabilities per cell (shared across B boxes)

4. **Multi-scale detection (YOLOv3+)**: Three detection heads at different scales (13x13, 26x26, 52x52) for large, medium, small objects

5. **Anchor-based regression**: Predict offsets from anchor boxes, not absolute coordinates

6. **NMS post-processing**: Apply NMS to remove duplicate detections across all cells

**Speed vs R-CNN**: YOLO processes the full image once; R-CNN runs a classifier ~2000 times (once per proposal). YOLO: ~25ms; R-CNN: ~49 seconds per image.

### Q5: What is the receptive field and why does it matter?

**Answer**:

The **receptive field** is the region of the input image that influences a given activation in a feature map.

**Why it matters**:
1. **Object size**: To detect a full object (e.g., a face), the receptive field must be at least as large as the object
2. **Context**: Tasks like semantic segmentation need large receptive fields to understand global context
3. **Architecture design**: Governs how deep a network needs to be for a given task

**How to increase receptive field**:
- Add more layers (linear growth per layer)
- Use pooling layers (multiplicative growth)
- Use dilated/atrous convolutions (exponential growth, used in DeepLab)
- Use larger kernels (but more expensive)

**Effective vs theoretical RF**: The theoretical RF grows with depth, but the *effective* RF (region that actually influences the output) is often much smaller -- central pixels contribute exponentially more than border pixels. This is why attention mechanisms (Transformers) can be more effective for global context.

In [ ]:
# Visualize effective receptive field concept
np.random.seed(42)

def effective_rf_simulation(n_layers, img_size=32, kernel=3):
    """Simulate how much each input pixel influences the center output."""
    # Initialize with uniform weights
    influence = np.zeros((img_size, img_size))
    center = img_size // 2
    influence[center, center] = 1.0  # Start from center output neuron

    for _ in range(n_layers):
        # Spread influence backward through a conv layer
        new_influence = np.zeros_like(influence)
        pad = kernel // 2
        padded = np.pad(influence, pad, mode='constant')
        for i in range(img_size):
            for j in range(img_size):
                # Each pixel receives influence from kernel neighborhood
                new_influence[i, j] = padded[i:i+kernel, j:j+kernel].sum() / (kernel*kernel)
        influence = new_influence / (new_influence.max() + 1e-10)
    return influence

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
depths = [1, 3, 7, 15]
for ax, d in zip(axes, depths):
    rf_map = effective_rf_simulation(d)
    im = ax.imshow(rf_map, cmap='hot')
    ax.set_title(f'{d} Conv Layers\n(3x3 each)', fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle('Effective Receptive Field Grows with Depth\n(brighter = more influence on center output neuron)',
             fontsize=12)
plt.tight_layout()
plt.show()

### Q6 through Q10 — More Interview Questions

**Q6: What is batch normalization and why is it useful in CNNs?**

Batch normalization normalizes activations within a mini-batch: `BN(x) = gamma * (x - mean) / std + beta` (where gamma, beta are learned).

**Benefits**:
- Reduces internal covariate shift (distribution of activations changes less between batches)
- Allows higher learning rates (activations stay in good range)
- Acts as regularizer (reduces need for dropout)
- Makes training less sensitive to weight initialization
- Placed between conv and activation function

**Interview gotcha**: BN behaves differently at train time (uses batch stats) vs inference time (uses running average stats). Forgetting to call `model.eval()` in PyTorch is a common bug.

---

**Q7: Explain the Inception module. What problem does it solve?**

The Inception module applies multiple filter sizes (1x1, 3x3, 5x5) in parallel and concatenates results.

**Problems solved**:
1. Uncertainty about optimal kernel size -- use all of them
2. Computational cost: 1x1 convolutions act as bottlenecks reducing channel depth before expensive 3x3/5x5 ops
3. Allows network to capture features at multiple scales simultaneously

**1x1 convolution role**: Projects high-dimensional feature maps to lower dimensions (channel-wise fully connected), reducing parameters by 10x before the larger convolutions.

---

**Q8: What is the difference between semantic segmentation and instance segmentation?**

| Type | Description | Example |
|------|-------------|--------|
| Classification | Image label | 'cat' |
| Object Detection | Bounding boxes + labels | Box around each cat |
| Semantic Segmentation | Per-pixel class label | Every 'cat pixel' = class 1 |
| Instance Segmentation | Per-pixel + instance | Cat1 pixels vs Cat2 pixels |

**Key models**:
- Semantic: FCN, U-Net, DeepLab
- Instance: Mask R-CNN (adds mask head to Faster R-CNN)

---

**Q9: What is dilated (atrous) convolution? When is it used?**

Dilated convolution inserts zeros between kernel weights, expanding the receptive field without increasing parameters or losing resolution.

- Dilation rate d=1: standard conv
- Dilation rate d=2: kernel sees every 2nd pixel
- Exponential dilation stacking: RF grows as 1, 2, 4, 8, 16...

**Used in**: DeepLab for semantic segmentation, WaveNet for audio, dense prediction tasks where downsampling destroys spatial resolution.

---

**Q10: What is the vanishing/exploding gradient problem and how do CNNs address it?**

**Vanishing**: Gradients become exponentially small in early layers. Causes: deep networks + sigmoid/tanh activations.

**Exploding**: Gradients become exponentially large. Causes: large weight initializations.

**Solutions in CNNs**:
1. **ReLU activation**: Gradient = 1 for positive inputs (no saturation)
2. **Batch normalization**: Keeps activations in healthy range
3. **Careful initialization**: He initialization (for ReLU), Xavier/Glorot
4. **Residual connections**: Identity path guarantees gradient flow
5. **Gradient clipping**: Caps gradient norm (for exploding)

### Q11 through Q15 — Advanced Questions

**Q11: How does depthwise separable convolution work and why is it efficient?**

Standard conv on (H, W, C_in) with C_out filters of size K: complexity = `H * W * K^2 * C_in * C_out`

Depthwise separable conv (MobileNet):
1. **Depthwise**: K x K conv applied independently to each channel: `H * W * K^2 * C_in`
2. **Pointwise**: 1x1 conv to mix channels: `H * W * C_in * C_out`

Total: `H * W * C_in * (K^2 + C_out)` vs `H * W * K^2 * C_in * C_out`

Reduction factor: `1/C_out + 1/K^2` -- for K=3, C_out=256: ~8-9x fewer operations.

**Used in**: MobileNet, Xception, EfficientNet.

---

**Q12: Explain the concept behind Attention in Vision Transformers (ViT).**

**ViT key idea**: Divide image into fixed patches (e.g., 16x16), embed each patch as a token, apply Transformer self-attention.

**Self-attention**: Each patch can attend to all other patches, enabling global context from the first layer (unlike CNNs which build context hierarchically through depth).

**Advantages over CNN**:
- No inductive bias for locality (can capture long-range dependencies)
- Scales better with data
- More interpretable attention maps

**Disadvantages**:
- Quadratic complexity in sequence length: O(N^2) where N = number of patches
- Needs large datasets (no locality inductive bias = harder to learn)
- Addressed by: Swin Transformer (windowed attention), DeiT (distillation)

---

**Q13: What is feature pyramid network (FPN) and why is it important for detection?**

FPN builds a top-down pathway that combines high-resolution (low-level) features with semantically rich (high-level) features.

**Problem it solves**: Small objects are best detected at high-resolution feature maps but those lack semantic context. Large objects need low-resolution maps with global context.

**Architecture**: Bottom-up (backbone), top-down path (upsampling + lateral connections from same-scale backbone features), prediction at every level.

**Used in**: RetinaNet, Mask R-CNN, most modern detectors.

---

**Q14: Explain IoU, GIoU, DIoU, and CIoU -- why were the improved versions needed?**

- **IoU**: Standard overlap. Problem: when boxes don't overlap, gradient = 0 regardless of distance.
- **GIoU** (Generalized IoU): Adds penalty based on smallest enclosing box: `GIoU = IoU - (C - U)/C`. Provides gradient when IoU=0.
- **DIoU** (Distance IoU): Adds penalty on center point distance: `DIoU = IoU - d^2/c^2`. Converges faster.
- **CIoU** (Complete IoU): Adds aspect ratio consistency term. Best training stability.

**Used in**: YOLOv5+, modern detectors use CIoU loss instead of MSE for bounding box regression.

---

**Q15: How do you handle class imbalance in object detection?**

In detection, background anchors vastly outnumber foreground: ratio can be 1000:1.

**Solutions**:
1. **Hard Negative Mining**: Select hardest (highest loss) negatives only, use fixed foreground:background ratio (e.g., 1:3)
2. **Focal Loss (RetinaNet)**: `FL(p) = -(1-p)^gamma * log(p)`. Reduces loss for easy negatives, focuses on hard examples. Gamma=2 standard.
3. **Oversampling**: Repeat minority class examples
4. **Balanced sampling**: Sample equal class examples per batch
5. **Weighted loss**: Scale loss by inverse class frequency

Focal Loss enabled one-stage detectors (RetinaNet) to match two-stage (Faster R-CNN) accuracy.

In [ ]:
# Visualize Focal Loss vs Cross-Entropy
def cross_entropy(p):
    return -np.log(p + 1e-10)

def focal_loss(p, gamma=2.0):
    return -(1 - p)**gamma * np.log(p + 1e-10)

probs = np.linspace(0.01, 0.99, 200)
ce = cross_entropy(probs)
fl1 = focal_loss(probs, gamma=0.5)
fl2 = focal_loss(probs, gamma=2.0)
fl3 = focal_loss(probs, gamma=5.0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(probs, ce, 'k-', linewidth=2.5, label='CE (gamma=0)')
ax1.plot(probs, fl1, '--', linewidth=2, label='Focal (gamma=0.5)')
ax1.plot(probs, fl2, '-', linewidth=2, label='Focal (gamma=2.0)')
ax1.plot(probs, fl3, '-.', linewidth=2, label='Focal (gamma=5.0)')
ax1.set_xlabel('Predicted Probability p')
ax1.set_ylabel('Loss')
ax1.set_ylim(0, 5)
ax1.set_title('Focal Loss vs Cross-Entropy')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.axvline(0.7, color='gray', linestyle=':', label='easy example')

# Modulating factor (1-p)^gamma
for gamma, color in [(0.5, 'steelblue'), (2.0, 'tomato'), (5.0, 'seagreen')]:
    modulating = (1 - probs)**gamma
    ax2.plot(probs, modulating, linewidth=2, color=color, label=f'gamma={gamma}')
ax2.set_xlabel('Predicted Probability p')
ax2.set_ylabel('Modulating Factor (1-p)^gamma')
ax2.set_title('Focal Loss: Modulating Factor\n(suppresses easy example loss)')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.suptitle('Focal Loss for Class Imbalance in Detection (Q15)', fontsize=12)
plt.tight_layout()
plt.show()
print('Key insight: for easy examples (p=0.9), modulating factor (gamma=2) = (0.1)^2 = 0.01')
print('Loss is 100x smaller for easy examples -- network focuses on hard negatives.')

---
## Summary: CV Interview Cheat Sheet

### Architecture Parameters Quick Reference

| Formula | Description |
|---------|-------------|
| `floor((W - K + 2P) / S) + 1` | Output size after conv |
| `RF_n = RF_{n-1} + (K-1) * stride_product` | Receptive field growth |
| `Params in conv = K*K*C_in*C_out + C_out` | Conv layer parameter count |
| `Depthwise savings = 1/C_out + 1/K^2` | MobileNet efficiency factor |
| `IoU = Intersection / Union` | Overlap metric |
| `NMS keeps highest-score, removes IoU > threshold` | Duplicate removal |

### Key Numbers to Memorize
- ResNet-50: 25.6M params, top-5 error 5.25%
- VGG-16: 138M params (mostly in FC layers)
- 2x 3x3 convs = same RF as 1x 5x5, but fewer params
- YOLO single pass: ~25ms; sliding window: seconds
- HOG: 128D descriptor per keypoint; SIFT: 128D
- Standard IoU threshold: 0.5 (PASCAL VOC), 0.5:0.95 (COCO)

In [ ]:
# Final summary: benchmark comparison of approaches on digits dataset
print('='*60)
print('Final Benchmark: Digits Classification Comparison')
print('='*60)

approaches = {}
# Raw pixels + LogReg
clf1 = Pipeline([('sc', StandardScaler()),
                  ('clf', LogisticRegression(max_iter=500))])
scores1 = cross_val_score(clf1, X, y, cv=5)
approaches['Raw Pixels + LogReg'] = scores1

# HOG features + SVM
try:
    from skimage.feature import hog as sk_hog
    X_hog_full = np.array([sk_hog(img, orientations=8, pixels_per_cell=(2,2),
                                   cells_per_block=(1,1)) for img in digits.images])
except ImportError:
    X_hog_full = X_hog  # already computed above
clf2 = Pipeline([('sc', StandardScaler()),
                  ('clf', SVC(kernel='rbf', C=5))])
scores2 = cross_val_score(clf2, X_hog_full, y, cv=5)
approaches['HOG + SVM'] = scores2

# PCA + SVM
clf3 = Pipeline([('sc', StandardScaler()),
                  ('pca', PCA(32, whiten=True)),
                  ('clf', SVC(kernel='rbf', C=5))])
scores3 = cross_val_score(clf3, X, y, cv=5)
approaches['PCA + SVM'] = scores3

# Random forest
clf4 = Pipeline([('sc', StandardScaler()),
                  ('clf', RandomForestClassifier(200, random_state=42))])
scores4 = cross_val_score(clf4, X, y, cv=5)
approaches['Random Forest'] = scores4

for name, scores in approaches.items():
    print(f'{name:<25}: {scores.mean():.4f} +/- {scores.std():.4f}')

print('\nNote: CNNs on larger image datasets (ImageNet) achieve >99% top-5 accuracy.')
print('These traditional methods work well on simple 8x8 digits.')
print('For natural images at scale, deep CNNs dominate.')

fig, ax = plt.subplots(figsize=(10, 5))
names_f = list(approaches.keys())
means_f = [approaches[n].mean() for n in names_f]
stds_f = [approaches[n].std() for n in names_f]
colors_f = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
bars = ax.bar(names_f, means_f, yerr=stds_f, capsize=7, color=colors_f, alpha=0.85)
ax.set_ylim(0.92, 1.01)
ax.set_ylabel('5-fold CV Accuracy')
ax.set_title('CV Pipeline Comparison: Digits Dataset')
for bar, m in zip(bars, means_f):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{m:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
## Notebook Complete

This notebook covered:

1. **Image Fundamentals** -- matrix representation, channels, basic operations, histogram equalization, edge detection
2. **Traditional CV** -- HOG features, SIFT/scale-space, augmentation, template matching, limitations
3. **CNN Architecture** -- convolution math, pooling, receptive fields, LeNet->ResNet progression, skip connections
4. **Object Detection** -- sliding window, anchors, YOLO, IoU, NMS, mAP, focal loss
5. **Practical Implementation** -- classification pipeline, PCA features, augmentation pipeline, transfer learning
6. **15 Interview Q&A** -- covering convolution, pooling, ResNet, YOLO, receptive fields, BN, Inception, segmentation, ViT, FPN, IoU variants, class imbalance

### Next Steps
- Implement a mini-CNN from scratch in NumPy (forward + backward pass)
- Train a real CNN on CIFAR-10 with PyTorch/TensorFlow
- Study Transformer-based vision models (ViT, Swin Transformer, DINO)
- Practice object detection with COCO dataset